<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных о горах

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку географических данных с помощью pandas.

**Данные:**
- `mountains.csv` — информация о горах мира: название, координаты, высота и тип горных пород

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файл с данными о горах в pandas DataFrame
3. Очищаем и анализируем структуру данных (координаты, высота, материалы)
4. Выполняем быструю валидацию данных и смотрим статистику


## 🐱 [1] Клонируем репозиторий курса в Colab

In [7]:
# 🐱 Шаг 1. Клонируем ваш репозиторий в Colab

import os

if not os.path.exists("python-ai-AnastasiaKalyashova"):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git

%cd python-ai-AnastasiaKalyashova

print("✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-AnastasiaKalyashova")

/content/python-ai-AnastasiaKalyashova/python-ai-AnastasiaKalyashova
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-AnastasiaKalyashova


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [8]:
# 🐱 Шаг 2A. Чтение CSV-файла с данными о горах

import pandas as pd

df_mountains = pd.read_csv("data/mountains.csv")

print("✅ Загружено строк в df_mountains:", len(df_mountains))
print("\nПервые 3 строки данных:")
print(df_mountains.head(3))

✅ Загружено строк в df_mountains: 4390

Первые 3 строки данных:
                              mountain mountainLabel  \
0  http://www.wikidata.org/entity/Q513   Джомолунгма   
1  http://www.wikidata.org/entity/Q513   Джомолунгма   
2  http://www.wikidata.org/entity/Q524       Везувий   

                  coordinates  elevation rockMaterialLabel  
0  Point(86.925 27.988055555)    8848.86     горная порода  
1  Point(86.925 27.988055555)    8848.86               лёд  
2    Point(14.42919 40.82261)    1281.00            Тефрит  


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `mountain` с URL (ссылкой на объект Wikidata) — нам не нужна ссылка, нам достаточно названия горы.
- Столбцы `mountainLabel` и `rockMaterialLabel` содержат читаемые подписи (название горы и тип горной породы/материала).

В этом шаге мы:
- удалим столбец с URL Wikidata (`mountain`);
- переименуем `mountainLabel → mountain`, `rockMaterialLabel → rockMaterial`;
- приведём числовой столбец `elevation` (высота) к типу `float` (высота может содержать дробные значения, например 8848.86).

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(float)` — переводит столбец к дробному типу (для точного отображения высоты).

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа.


In [9]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# Удаляем технический столбец с URL Wikidata
df_mountains = df_mountains.drop(columns=["mountain"])

# Переименовываем столбцы с читаемыми подписями
df_mountains = df_mountains.rename(columns={
    "mountainLabel": "mountain",
    "rockMaterialLabel": "rockMaterial"
})

# Приводим высоту к числовому типу (дробное число)
df_mountains["elevation"] = pd.to_numeric(
    df_mountains["elevation"], errors="coerce"
).fillna(0).astype(float)

print("✅ Данные очищены и готовы к анализу")
print("\nТекущие столбцы:", list(df_mountains.columns))
print("\nПример данных после очистки:")
print(df_mountains.head(3))

✅ Данные очищены и готовы к анализу

Текущие столбцы: ['mountain', 'coordinates', 'elevation', 'rockMaterial']

Пример данных после очистки:
      mountain                 coordinates  elevation   rockMaterial
0  Джомолунгма  Point(86.925 27.988055555)    8848.86  горная порода
1  Джомолунгма  Point(86.925 27.988055555)    8848.86            лёд
2      Везувий    Point(14.42919 40.82261)    1281.00         Тефрит


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о горах:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по высоте (`elevation`) — минимальная, максимальная, средняя высота и т.д.

Для удобства используем функцию `show_info(df, name)`, чтобы компактно вывести информацию о таблице.


In [10]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных о горах

show_info(df_mountains, "Горы мира (df_mountains)")

print("\n📈 Статистика по высоте (elevation):")
print(df_mountains["elevation"].describe())

print("\n🪨 Распределение типов горных пород/материалов (rockMaterial):")
print(df_mountains["rockMaterial"].value_counts())


📊 Горы мира (df_mountains)
Размер: (4390, 4)
Столбцы: mountain, coordinates, elevation, rockMaterial

Первые строки:
      mountain                 coordinates  elevation   rockMaterial
0  Джомолунгма  Point(86.925 27.988055555)    8848.86  горная порода
1  Джомолунгма  Point(86.925 27.988055555)    8848.86            лёд
2      Везувий    Point(14.42919 40.82261)    1281.00         Тефрит
3      Монблан   Point(6.865 45.832777777)    4805.59         гранит
4      Монблан   Point(6.865 45.832777777)    4805.59          гнейс

📈 Статистика по высоте (elevation):
count     4390.000000
mean      1581.164787
std       1300.251027
min        -39.000000
25%        692.000000
50%       1233.900000
75%       2283.500000
max      16390.000000
Name: elevation, dtype: float64

🪨 Распределение типов горных пород/материалов (rockMaterial):
rockMaterial
известняк         834
песчаник          518
гранит            346
Мергель           301
Конгломерат       289
                 ... 
green tuff     

## ❄️ [4] Уникальный анализ: «многослойность» гор и «ледяной пояс» Земли

Ваши данные обладают редкой особенностью — **геологическая «многослойность»**: одна и та же гора может иметь несколько записей с разными типами пород и материалов. Например, Джомолунгма представлена двумя слоями: «горная порода» у основания и «лёд» на вершине.

В этом шаге мы исследуем два уникальных явления:

### 🗻 1. Геологическая «многослойность»
- Сколько **уникальных гор** скрыто в 4 390 записях?
- Какие горы имеют **наибольшее разнообразие материалов** (геологическая сложность)?
- Есть ли горы, представленные 5+ разными породами — признак сложной структуры?

### 🧊 2. «Ледяной пояс» Земли
Лёд не встречается равномерно — он концентрируется в определённом высотном диапазоне. Мы выявим:
- На какой **минимальной высоте** начинает формироваться постоянный ледник?
- В каком **диапазоне высот** лёд встречается чаще всего («ледяной пояс»)?
- Есть ли горы с льдом **ниже 4 000 м** — признак полярного климата?

> 💡 **Интересный факт**: В Гималаях ледники начинаются ~5 000 м, а в Антарктиде — уже на уровне моря. Анализ высоты появления льда поможет косвенно определить географическое положение гор!


In [12]:
# ❄️ Шаг 4. Анализ «многослойности» и «ледяного пояса»

print("=" * 70)
print("🏔️  ЧАСТЬ 1: Геологическая «многослойность» гор")
print("=" * 70)

# 1.1 Сравнение уникальных гор vs общего числа записей
total_records = len(df_mountains)
unique_mountains = df_mountains["mountain"].nunique()
print(f"\n📊 Всего записей: {total_records}")
print(f"📊 Уникальных гор (по названию): {unique_mountains}")
print(f"📊 Среднее число записей на гору: {total_records / unique_mountains:.2f}")
print(f"   → Это означает, что в среднем у каждой горы есть данные о {int(total_records / unique_mountains)} типах материалов!")

# 1.2 Распределение «многослойности»
layer_counts = df_mountains.groupby("mountain").size()
distribution = layer_counts.value_counts().sort_index()
print("\n📈 Распределение записей на гору:")
for layers, count in distribution.head(6).items():
    print(f"   {layers} слоя(ев): {count} гор(ы)")

# 1.3 Топ-10 гор с наибольшим разнообразием материалов
top_complex = (df_mountains.groupby("mountain")
               .agg({"rockMaterial": "nunique", "elevation": "first"})
               .sort_values("rockMaterial", ascending=False)
               .head(10))

print("\n🏆 Топ-10 гор с наибольшим разнообразием материалов:")
print(top_complex.reset_index().to_string(index=False))

print("\n" + "=" * 70)
print("🧊 ЧАСТЬ 2: «Ледяной пояс» Земли — где живёт лёд?")
print("=" * 70)

# 2.1 Поиск всех вариантов написания «лёд» (русский/английский/опечатки)
ice_variants = ["лёд", "лед", "ice", "Ice", "Лёд", "Лед"]
df_mountains["is_ice"] = df_mountains["rockMaterial"].astype(str).str.contains('|'.join(ice_variants), case=False, na=False)

ice_records = df_mountains[df_mountains["is_ice"]]
total_ice = len(ice_records)
print(f"\n❄️  Записей с льдом: {total_ice} из {total_records} ({total_ice/total_records*100:.1f}%)")

if total_ice > 0:
    # 2.2 Минимальная и максимальная высота с льдом
    min_ice = ice_records["elevation"].min()
    max_ice = ice_records["elevation"].max()
    print(f"   Минимальная высота с льдом: {min_ice:.0f} м")
    print(f"   Максимальная высота с льдом: {max_ice:.0f} м")

    # 2.3 Высотные диапазоны для анализа «ледяного пояса»
    bins = [0, 2000, 4000, 5000, 6000, 7000, 8000, 10000]
    labels = ["0–2 км", "2–4 км", "4–5 км", "5–6 км", "6–7 км", "7–8 км", "8+ км"]

    df_mountains["height_bin"] = pd.cut(df_mountains["elevation"], bins=bins, labels=labels, right=False)
    ice_by_bin = (df_mountains.groupby("height_bin")
                  .agg(total=("is_ice", "size"), ice=("is_ice", "sum"))
                  .assign(ice_pct=lambda x: (x["ice"] / x["total"] * 100).round(1))
                  .sort_index())

    print("\n📊 Распределение льда по высотным диапазонам:")
    print(ice_by_bin[["total", "ice", "ice_pct"]].rename(columns={
        "total": "Всего записей",
        "ice": "С льдом",
        "ice_pct": "% с льдом"
    }).to_string())

    # 2.4 Определение «ледяного пояса» — диапазон с максимальной концентрацией льда
    peak_bin = ice_by_bin["ice_pct"].idxmax()
    peak_value = ice_by_bin.loc[peak_bin, "ice_pct"]
    print(f"\n🎯 «Ледяной пояс» Земли: {peak_bin} — здесь лёд встречается в {peak_value}% записей!")

    # 2.5 Интересный факт: горы с льдом ниже 4000 м (полярные регионы?)
    low_ice = ice_records[ice_records["elevation"] < 4000]
    if len(low_ice) > 0:
        print(f"\n🔍 Необычные находки: {len(low_ice)} записей с льдом ниже 4000 м")
        print("   Примеры:")
        for _, row in low_ice.head(5).iterrows():
            print(f"   • {row['mountain']} ({row['elevation']:.0f} м) — {row['rockMaterial']}")
else:
    print("   ⚠️  Лёд не обнаружен в данных. Проверьте написание в столбце rockMaterial.")

# Очистка временных столбцов
df_mountains.drop(columns=["is_ice", "height_bin"], inplace=True, errors="ignore")

🏔️  ЧАСТЬ 1: Геологическая «многослойность» гор

📊 Всего записей: 4390
📊 Уникальных гор (по названию): 2832
📊 Среднее число записей на гору: 1.55
   → Это означает, что в среднем у каждой горы есть данные о 1 типах материалов!

📈 Распределение записей на гору:
   1 слоя(ев): 1814 гор(ы)
   2 слоя(ев): 638 гор(ы)
   3 слоя(ев): 308 гор(ы)
   4 слоя(ев): 35 гор(ы)
   5 слоя(ев): 14 гор(ы)
   6 слоя(ев): 13 гор(ы)

🏆 Топ-10 гор с наибольшим разнообразием материалов:
       mountain  rockMaterial  elevation
Puig de l'Àliga             8      464.0
     La Creueta             7      813.0
      El Tossal             7     1551.0
    Tossal Gros             6      867.0
     el Cogulló             6     1002.0
       El Pedró             6     1448.0
      Roca Roja             6     1239.0
  Santa Bàrbara             6     1613.0
     Montpedrós             5      348.0
     Montllobar             5     1104.0

🧊 ЧАСТЬ 2: «Ледяной пояс» Земли — где живёт лёд?

❄️  Записей с льдом: 3 из 4390

/tmp/ipython-input-3708270760.py:55: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ice_by_bin = (df_mountains.groupby("height_bin")


## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Посмотрели базовую статистику по бюджету (`capital_cost`)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨